## 1. Setup & Load Raw Data

In [1]:
import pandas as pd

transactions = pd.read_csv("../data/raw/finlora_transactions.csv")
accounts = pd.read_csv("../data/raw/finlora_accounts.csv")

## 2. Initial Inspection

In [2]:
transactions.info()


<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   transaction_id              126000 non-null  str    
 1   account_id                  126000 non-null  str    
 2   account_type                126000 non-null  str    
 3   kyc_tier                    126000 non-null  str    
 4   timestamp                   126000 non-null  str    
 5   day_of_week                 126000 non-null  str    
 6   hour_of_day                 126000 non-null  int64  
 7   description                 126000 non-null  str    
 8   merchant_name               97390 non-null   str    
 9   merchant_category           126000 non-null  str    
 10  channel                     126000 non-null  str    
 11  amount                      126000 non-null  float64
 12  currency                    126000 non-null  str    
 13  amount_to_avg_ratio      

### Anomalies from Transcations Data
Missing values:

merchant_name: 97,390 / 126,000 non-null → ~28,610 missing (~23%)

### Anomalies from Transactions Table ||
*** timestamp is str, not a datetime — we'll need to fix that - pd.to_datetime() 

In [3]:
accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   account_id                   7200 non-null   str    
 1   account_holder_name          7200 non-null   str    
 2   account_type                 7200 non-null   str    
 3   home_country                 7200 non-null   str    
 4   currency                     7200 non-null   str    
 5   kyc_tier                     7200 non-null   str    
 6   account_created_date         7200 non-null   str    
 7   personal_spend_baseline_usd  7200 non-null   float64
dtypes: float64(1), str(7)
memory usage: 450.2 KB


### Anomalies from Account Data
Missing Value - device_id: 111,666 / 126,000 non-null → ~14,334 missing (~11%)

*** account_created_date (in accounts) is also str — fixing needed, can impact account "age" as a feature.

### Observation: Columns that exist in both Transactions and Accounts Table:
account_type, 
home_country, 
kyc_tier 

###### Data types that will need conversion before feature engineering:

### Transactions - timestamp is str, not a datetime — we'll need to fix that - pd.to_datetime() 

### Accounts - account_created_date (in accounts) is also str — fixing needed, can impact account "age" as a feature.

In [4]:
# Further look at Transactions table(to mitigate truncation)

In [5]:
print(transactions.columns.tolist())
print("\nMissing values per column:\n", transactions.isnull().sum())
print("\n\nSample rows:")
transactions.head(3)


['transaction_id', 'account_id', 'account_type', 'kyc_tier', 'timestamp', 'day_of_week', 'hour_of_day', 'description', 'merchant_name', 'merchant_category', 'channel', 'amount', 'currency', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'transaction_velocity_1h', 'transaction_country', 'home_country', 'is_cross_border', 'device_id', 'is_new_device', 'account_age_days', 'status', 'is_fraud']

Missing values per column:
 transaction_id                    0
account_id                        0
account_type                      0
kyc_tier                          0
timestamp                         0
day_of_week                       0
hour_of_day                       0
description                       0
merchant_name                 28610
merchant_category                 0
channel                           0
amount                            0
currency                          0
amount_to_avg_ratio               0
avg_transaction_amount_30d        0
transaction_velocity_1h        

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0


In [6]:
transactions.head(3)

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0


###### What we can gather so far:

### device_id / is_new_device missing together, and not randomly. Look at your sample rows: rows 0 and 1 are CNP PURCHASE and WEB PURCHASE — both missing device info. Row 2 is a MOBILE PURCHASE — has a device ID. That's not random missingness (MCAR) — it's structural: card-not-present and web transactions probably just don't have a device to track, unlike mobile app transactions. That's an important distinction, because it changes the fix:

If missing were random, you might impute or drop rows.

Considering that it looks structurally (driven by channel), the right fix is for us to keep the missing values. Filling it wrong could quietly destroy integrity of real fraud signals.

### merchant_name missing (~23%) — Pattern not confirmed yet (Does it correlates with channel or merchant_category?). We should check before knowing what to do.

### Duplicate columns across tables (account_type, kyc_tier, home_country in both transactions and accounts) — need to verify they actually agree before we join - to duplication of columns that both look valid but disagree.

### status has values like Completed and Declined — worth checking: does is_fraud only ever "1" on Completed transactions, or can a Declined transaction still be labeled fraud (e.g. a blocked fraud attempt)?

****** This single block will settle: how to handle both missing-value columns, whether the join is safe to do naively or needs conflict-resolution logic, whether status needs to feed into your target definition, and how imbalanced your fraud classes actually are (which will directly shape your Step 5 modeling choices — that's why the brief already calls for class-balanced Logistic Regression and balanced subsampling in Random Forest, but we should see the real number).

## 3. Diagnosing Missing Values & Join Integrity

In [7]:
# Are 'device_id' missing values impacted by 'channel'
print("Missing device_id by channel:")
print(transactions.groupby('channel')['device_id'].apply(lambda x: x.isnull().mean()))

# Are the missing values in 'merchant_name' impacted by 'channel' or 'merchant_category'
print("\nMissing merchant_name by channel:")
print(transactions.groupby('channel')['merchant_name'].apply(lambda x: x.isnull().mean()))

# Does 'account_type' | 'kyc_tier' | 'home_country' tally between the two tables
merged_check = transactions.merge(accounts, on='account_id', suffixes=('_txn', '_acct'))
print("\naccount_type mismatches:", (merged_check['account_type_txn'] != merged_check['account_type_acct']).sum())
print("kyc_tier mismatches:", (merged_check['kyc_tier_txn'] != merged_check['kyc_tier_acct']).sum())
print("home_country mismatches:", (merged_check['home_country_txn'] != merged_check['home_country_acct']).sum())

Missing device_id by channel:
channel
API/Integration     0.112360
Card Not Present    0.110492
Card Present        0.112069
Mobile App          0.114406
USSD                0.114056
Web Dashboard       0.116055
Name: device_id, dtype: float64

Missing merchant_name by channel:
channel
API/Integration     0.279141
Card Not Present    0.208989
Card Present        0.214390
Mobile App          0.209348
USSD                0.206961
Web Dashboard       0.249793
Name: merchant_name, dtype: float64

account_type mismatches: 0
kyc_tier mismatches: 0
home_country mismatches: 0


## 4. Checking for Data Leakage & Class Balance

In [8]:
# Is 'status' correlated with 'is_fraud'
print("\nis_fraud by status:")
print(transactions.groupby('status')['is_fraud'].agg(['count', 'sum', 'mean']))

# fraud rate — Overview 
print("\nOverall fraud rate:", transactions['is_fraud'].mean())
print(transactions['is_fraud'].value_counts())


is_fraud by status:
            count   sum      mean
status                           
Completed  116703  2032  0.017412
Declined     5629    99  0.017587
Reversed     3668  1221  0.332879

Overall fraud rate: 0.026603174603174604
is_fraud
0    122648
1      3352
Name: count, dtype: int64


#### The finding: The above signifies a leakage risk

Completed: 1.74% fraud
Declined: 1.76% fraud
Reversed: 33.3% fraud — almost 19x higher

"Reversed" are likely results from fraud detection such as chargebacks or manual reversal after an analyst or the cardholder disputed the transaction. Meanin that 'status' may not be known when scoring a transaction. If included as a feature in a model, it misleads predictions by learning that "Reversed" indicates fraud, leading to inflated testing metrics but poor real-world performance due to data leakage. 

### Decision - 'Status' will NOT be considered as a model feature. It is an artifact and not a predictive signal. It'll be Kept it in the dataset for EDA, but omitted from the modeling feature set. 

### Deduction from the Numbers:

device_id missingness is NOT sufficiently explained by channel. Every channel sits between ~11–11.6% missing which is a tight band. 

merchant_name missingness has a mild channel pattern but it's not dramatic — API/Integration (28%) and Web Dashboard (25%) run a bit higher than Card Not Present/Present, Mobile App, USSD (~21%). This does NOT give a clean structural split.

Table join is CLEAN and SAFE — ZERO MISMATECHES on account_type, kyc_tier, home_country between the two tables. Confidently, we can drop the duplicated columns from one side.

Fraud rate: 3,352 / 126,000 ≈ 2.66%. This signifies a severe classification imbalance problem — now confirmed with a real number. Our brief specifies class_weight='balanced' for Logistic Regression and balanced subsampling for Random Forest -- Accuracy would be a useless metric here when - (a model that predicts "not fraud" for everything scores 97.3% accuracy and is worthless), and why precision/recall/F1/ROC-AUC are the right lens.

## 5. Clean Data & Join Transactions with Accounts

In [9]:
import pandas as pd #Importing panada again

In [10]:
transactions = pd.read_csv("../data/raw/finlora_transactions.csv")
accounts = pd.read_csv("../data/raw/finlora_accounts.csv") #Loading raw data again (transctions & accounts csv files)

In [11]:
transactions['timestamp'] = pd.to_datetime(transactions['timestamp']) #Transctions [Cleaning] = timestamp to to_datetime

In [12]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   transaction_id              126000 non-null  str           
 1   account_id                  126000 non-null  str           
 2   account_type                126000 non-null  str           
 3   kyc_tier                    126000 non-null  str           
 4   timestamp                   126000 non-null  datetime64[us]
 5   day_of_week                 126000 non-null  str           
 6   hour_of_day                 126000 non-null  int64         
 7   description                 126000 non-null  str           
 8   merchant_name               97390 non-null   str           
 9   merchant_category           126000 non-null  str           
 10  channel                     126000 non-null  str           
 11  amount                      126000 non-null  float

In [13]:
accounts['account_created_date'] = pd.to_datetime(accounts['account_created_date']) #Account [Cleaning] = timestamp to to_datetime

In [14]:
accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   account_id                   7200 non-null   str           
 1   account_holder_name          7200 non-null   str           
 2   account_type                 7200 non-null   str           
 3   home_country                 7200 non-null   str           
 4   currency                     7200 non-null   str           
 5   kyc_tier                     7200 non-null   str           
 6   account_created_date         7200 non-null   datetime64[us]
 7   personal_spend_baseline_usd  7200 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 450.2 KB


In [15]:
# --- Join transactions to accounts on account_id ---
# account_type, kyc_tier, home_country are present on both tables and confirmed
# we are droping the accounts-side versions and sticking to the transactions table and its transaction-time version
accounts_to_merge = accounts.drop(columns=['account_type', 'kyc_tier', 'home_country'])

df = transactions.merge(accounts_to_merge, on='account_id', how='left')

In [16]:
# --- Handling our missing values for merchant_name ---
# We replace the empty cells under the merchant_names column with 'Unknown'.
df['merchant_name'] = df['merchant_name'].fillna('Unknown')

In [17]:
# --- Handling our missing values for 'device_id' ---
# We create a new column - 'has_device_on_record' without altering our 'device_id' column;
# such that instances where we have "no device on record" = 0
# such that instances where we have "device on record" = 1

df['has_device_on_record'] = df['device_id'].notna().astype(int)

In [18]:
# --- Handling our missing values for 'device_id' ---
# In our 'device_id' column, we replace all empty cells with 'NO_DEVICE_RECORDED'.
df['device_id'] = df['device_id'].fillna('NO_DEVICE_RECORDED')

## Quick Sanity Checks

In [19]:
df.shape #final data shape check

(126000, 29)

In [20]:
df.isnull().sum()[df.isnull().sum() > 0] #Remaining nulls check - we don't expect to see 'device_id' or merchant_name

is_new_device    14334
dtype: int64

In [21]:
df.duplicated().sum() #Duplicate rows check

np.int64(0)

## Saving our Cleaned and Joined Dataset

In [22]:
# --- Save cleaned, joined dataset ---
df.to_csv("../data/processed/finlora_cleaned.csv", index=False) #save to processed folder, remove pandas column from file

In [23]:
print("\nSaved to data/processed/finlora_cleaned.csv") #checking if our new file was successfully saved


Saved to data/processed/finlora_cleaned.csv
